imports

In [8]:
import json
import joblib
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

In [11]:
BASE_DIR = Path(r"D:\VU project CS619\ML based Cyber Attack Detection\ML model training\next-level experiment model training")

BALANCED_PATH = BASE_DIR / "datasets" / "processed" / "master_dataset_balanced_v1.csv"

MODELS_DIR = BASE_DIR / "models"
RF_DIR = MODELS_DIR / "random_forest"
SVM_DIR = MODELS_DIR / "svm"
IF_DIR = MODELS_DIR / "isolation_forest"

print("Balanced dataset path:", BALANCED_PATH)
print("Exists:", BALANCED_PATH.exists())

Balanced dataset path: D:\VU project CS619\ML based Cyber Attack Detection\ML model training\next-level experiment model training\datasets\processed\master_dataset_balanced_v1.csv
Exists: True


In [12]:
balanced_raw_df = pd.read_csv(BALANCED_PATH)

print("Loaded dataset shape:", balanced_raw_df.shape)
print("Columns:", balanced_raw_df.columns.tolist())
balanced_raw_df.head()

Loaded dataset shape: (53379, 15)
Columns: ['no_', 'time', 'source', 'destination', 'protocol', 'source_port', 'length', 'tcp_flags', 'dst_port', 'stream_id', 'delta_time', 'info', 'label', 'traffic_type', 'source_file']


,no_,time,source,destination,protocol,source_port,length,tcp_flags,dst_port,stream_id,delta_time,info,label,traffic_type,source_file
0,1,0.000000,192.168.48.133,192.168.48.1,TCP,48714.0,74,0x0002,53.0,0.0,NaN,48714 > 53 [SYN] Seq=0 Win=64240 Len=0 MSS=1...,0,normal,normal_ftp_transfer.csv
1,2,0.032710,192.168.48.130,192.168.48.1,DNS,49170.0,81,NaN,53.0,NaN,0.032710,Standard query 0x3aed A 2.debian.pool.ntp.org,0,normal,normal_ftp_transfer.csv
2,3,0.032792,192.168.48.130,192.168.48.1,DNS,49170.0,81,NaN,53.0,NaN,0.000082,Standard query 0x09e8 AAAA 2.debian.pool.ntp.org,0,normal,normal_ftp_transfer.csv
3,4,1.031463,192.168.48.133,192.168.48.1,TCP,48714.0,74,0x0002,53.0,0.0,0.998671,[TCP Retransmission] 48714 > 53 [SYN] Seq=0 ...,0,normal,normal_ftp_transfer.csv
4,5,3.047663,192.168.48.133,192.168.48.1,TCP,48714.0,74,0x0002,53.0,0.0,2.016200,[TCP Retransmission] 48714 > 53 [SYN] Seq=0 ...,0,normal,normal_ftp_transfer.csv


In [13]:
print(balanced_raw_df.dtypes)

print("\nprotocol sample:", balanced_raw_df["protocol"].head(10).tolist())
print("\ntcp_flags sample:", balanced_raw_df["tcp_flags"].head(10).tolist())
print("\ninfo sample:", balanced_raw_df["info"].head(10).tolist())

no_               int64
time            float64
source           object
destination      object
protocol         object
source_port     float64
length            int64
tcp_flags        object
dst_port        float64
stream_id       float64
delta_time      float64
info             object
label             int64
traffic_type     object
source_file      object
dtype: object

protocol sample: ['TCP', 'DNS', 'DNS', 'TCP', 'TCP', 'DNS', 'DNS', 'TCP', 'TCP', 'DNS']

tcp_flags sample: ['0x0002', nan, nan, '0x0002', '0x0002', nan, nan, '0x0002', '0x0002', nan]

info sample: ['48714  >  53 [SYN] Seq=0 Win=64240 Len=0 MSS=1460 SACK_PERM TSval=3386958897 TSecr=0 WS=128', 'Standard query 0x3aed A 2.debian.pool.ntp.org', 'Standard query 0x09e8 AAAA 2.debian.pool.ntp.org', '[TCP Retransmission] 48714  >  53 [SYN] Seq=0 Win=64240 Len=0 MSS=1460 SACK_PERM TSval=3386959929 TSecr=0 WS=128', '[TCP Retransmission] 48714  >  53 [SYN] Seq=0 Win=64240 Len=0 MSS=1460 SACK_PERM TSval=3386961945 TSecr=0 WS=128',

In [15]:
encoders = {}

for col in ["protocol", "tcp_flags", "info"]:
    le = LabelEncoder()
    balanced_raw_df[col] = balanced_raw_df[col].astype(str).fillna("unknown")
    le.fit(balanced_raw_df[col])
    encoders[col] = le

print("Encoders rebuilt successfully.")
print("Saved columns:", list(encoders.keys()))

Encoders rebuilt successfully.
Saved columns: ['protocol', 'tcp_flags', 'info']


In [16]:
label_encoders_path = MODELS_DIR / "label_encoders.pkl"
joblib.dump(encoders, label_encoders_path)

print("Saved label encoders to:")
print(label_encoders_path)

Saved label encoders to:
D:\VU project CS619\ML based Cyber Attack Detection\ML model training\next-level experiment model training\models\label_encoders.pkl


In [17]:
encoder_report = {}

for col, le in encoders.items():
    encoder_report[col] = {
        "classes": le.classes_.tolist()
    }

encoder_report_path = MODELS_DIR / "label_encoder_classes.json"

with open(encoder_report_path, "w", encoding="utf-8") as f:
    json.dump(encoder_report, f, indent=4)

print("Saved encoder class report to:")
print(encoder_report_path)

Saved encoder class report to:
D:\VU project CS619\ML based Cyber Attack Detection\ML model training\next-level experiment model training\models\label_encoder_classes.json


In [18]:
preprocessing_config = {
    "project_name": "ML Based Cyber Attack Detection",
    "dataset_used_for_encoder_rebuild": str(BALANCED_PATH),
    "final_feature_order": [
        "protocol",
        "source_port",
        "length",
        "tcp_flags",
        "dst_port",
        "delta_time",
        "info"
    ],
    "target_column": "label",
    "categorical_columns": [
        "protocol",
        "tcp_flags",
        "info"
    ],
    "numeric_columns": [
        "source_port",
        "length",
        "dst_port",
        "delta_time"
    ],
    "dropped_columns_before_training": [
        "no_",
        "source",
        "destination",
        "traffic_type",
        "source_file",
        "time",
        "stream_id"
    ],
    "model_artifacts": {
        "random_forest": {
            "model_path": str(RF_DIR / "rf_behavioral_final.pkl"),
            "features_path": str(RF_DIR / "features.txt"),
            "scaler_required": False
        },
        "svm": {
            "model_path": str(SVM_DIR / "svm_behavioral_v1.pkl"),
            "features_path": str(SVM_DIR / "features.txt"),
            "scaler_required": True,
            "scaler_path": str(SVM_DIR / "svm_scaler_v1.pkl")
        },
        "isolation_forest": {
            "model_path": str(IF_DIR / "iforest_behavioral_v1.pkl"),
            "features_path": str(IF_DIR / "features.txt"),
            "scaler_required": True,
            "scaler_path": str(IF_DIR / "iforest_scaler_v1.pkl")
        }
    }
}

config_path = MODELS_DIR / "preprocessing_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(preprocessing_config, f, indent=4)

print("Saved preprocessing config to:")
print(config_path)

Saved preprocessing config to:
D:\VU project CS619\ML based Cyber Attack Detection\ML model training\next-level experiment model training\models\preprocessing_config.json


In [19]:
final_feature_columns = [
    "protocol",
    "source_port",
    "length",
    "tcp_flags",
    "dst_port",
    "delta_time",
    "info"
]

for model_dir in [RF_DIR, SVM_DIR, IF_DIR]:
    with open(model_dir / "features.txt", "w", encoding="utf-8") as f:
        for col in final_feature_columns:
            f.write(col + "\n")

print("Feature files refreshed for all models.")

Feature files refreshed for all models.


In [20]:
required_files = [
    MODELS_DIR / "label_encoders.pkl",
    MODELS_DIR / "label_encoder_classes.json",
    MODELS_DIR / "preprocessing_config.json",

    RF_DIR / "rf_behavioral_final.pkl",
    RF_DIR / "features.txt",

    SVM_DIR / "svm_behavioral_v1.pkl",
    SVM_DIR / "svm_scaler_v1.pkl",
    SVM_DIR / "features.txt",

    IF_DIR / "iforest_behavioral_v1.pkl",
    IF_DIR / "iforest_scaler_v1.pkl",
    IF_DIR / "features.txt",
]

print("Artifact verification:\n")
for fp in required_files:
    print(f"{fp.name}: {'FOUND' if fp.exists() else 'MISSING'}")

Artifact verification:

label_encoders.pkl: FOUND
label_encoder_classes.json: FOUND
preprocessing_config.json: FOUND
rf_behavioral_final.pkl: FOUND
features.txt: FOUND
svm_behavioral_v1.pkl: FOUND
svm_scaler_v1.pkl: FOUND
features.txt: FOUND
iforest_behavioral_v1.pkl: FOUND
iforest_scaler_v1.pkl: FOUND
features.txt: FOUND


In [ ]:
just tested with predictor file and sample raw 

In [21]:
from pathlib import Path
import pandas as pd
import sys

BASE_DIR = r"D:\VU project CS619\ML based Cyber Attack Detection\ML model training\next-level experiment model training"
sys.path.append(str(Path(BASE_DIR) / "utils"))

from predictor import TrafficPredictor

predictor = TrafficPredictor(BASE_DIR)

test_csv = Path(BASE_DIR) / "datasets" / "processed" / "master_dataset_balanced_v1.csv"
df_test = pd.read_csv(test_csv).head(20)

result = predictor.predict(df_test, model_name="random_forest")
print(result[["protocol", "source_port", "length", "prediction", "prediction_label"]].head(10))

  protocol  source_port  length  prediction prediction_label
0      TCP      48714.0      74           0           normal
1      DNS      49170.0      81           0           normal
2      DNS      49170.0      81           0           normal
3      TCP      48714.0      74           0           normal
4      TCP      48714.0      74           0           normal
5      DNS      54557.0      93           0           normal
6      DNS      54557.0      93           0           normal
7      TCP      48714.0      74           0           normal
8      TCP      35250.0      74           0           normal
9      DNS      54557.0      93           0           normal
